# Fabric Workspaces — List & Create

This notebook uses the **Microsoft Fabric REST API** to:

1. List all workspaces visible to the running identity
2. Create a set of workspaces from a configuration array, optionally assigning a security group

Authentication is handled via `notebookutils`, so the notebook runs as the **authenticated identity executing it** (user or service principal configured on the notebook). No client secrets required.

> Make sure the running identity has permission to create workspaces on the target capacity.

In [ ]:
# ----------------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------------
import requests

# Workspace settings
CAPACITY_ID = "<your-capacity-id>"
GROUP_ID    = ""            # Security group ID for ACL. Leave empty to skip role assignment.
ROLE        = "Member"      # Admin | Member | Contributor | Viewer

# Workspaces to create
WORKSPACES = [
    {"name": "WS - Sales",     "description": "Sales analytics workspace"},
    {"name": "WS - Marketing", "description": "Marketing analytics workspace"},
    {"name": "WS - Finance",   "description": "Finance analytics workspace"},
]

WORKSPACES_URL = "https://api.fabric.microsoft.com/v1/workspaces"

# Get token for the running identity via notebookutils
token = notebookutils.credentials.getToken("pbi")

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

In [ ]:
# ----------------------------------------------------------------------------
# LIST WORKSPACES
# ----------------------------------------------------------------------------
response = requests.get(WORKSPACES_URL, headers=headers)
response.raise_for_status()

workspaces = response.json().get("value", [])
print(f"Found {len(workspaces)} workspaces\n")

for ws in workspaces:
    capacity_id = ws.get("capacityId", "N/A")
    print(f"* {ws.get('displayName')} (ID: {ws.get('id')}, Capacity ID: {capacity_id})")

In [ ]:
# ----------------------------------------------------------------------------
# CREATE WORKSPACES
# ----------------------------------------------------------------------------
for ws in WORKSPACES:
    body = {
        "displayName": ws["name"],
        "description": ws["description"],
        "capacityId":  CAPACITY_ID,
    }

    response = requests.post(WORKSPACES_URL, headers=headers, json=body)
    response.raise_for_status()

    workspace_id = response.json()["id"]
    print(f"* Created '{ws['name']}' (ID: {workspace_id})")

    # Assign security group with the configured role (skip if no group provided)
    if GROUP_ID:
        acl_body = {
            "principal": {"id": GROUP_ID, "type": "Group"},
            "role": ROLE,
        }
        acl_url = f"{WORKSPACES_URL}/{workspace_id}/roleAssignments"
        acl_response = requests.post(acl_url, headers=headers, json=acl_body)
        acl_response.raise_for_status()
        print(f"  - Assigned group {GROUP_ID} as {ROLE}")